# Unit 8: TensorBoard — 深度学习可视化利器

## 学习目标
- 理解 TensorBoard 的核心功能和使用场景
- 掌握 `SummaryWriter` 的各种日志记录 API
- 学会可视化损失曲线、准确率、权重分布、梯度流
- 学会记录模型计算图和图像
- 掌握 `add_hparams` 进行超参数对比实验
- 在 Jupyter 中内嵌 TensorBoard
- 实战：用 TensorBoard 完整跟踪 CIFAR-10 训练

## 8.1 为什么需要 TensorBoard？

训练深度学习模型时你面临的问题：
- 损失曲线靠 `print` 看？几十个 epoch 后根本看不清趋势
- 模型结构复杂？靠想象画不出来
- 梯度消失了？不知道什么时候开始消失的
- 试了 10 组超参数？手动对比太痛苦

**TensorBoard 的解决方案**：一个统一的 Web 界面，实时可视化训练过程中的**一切**。

它原本是 TensorFlow 的一部分，但 PyTorch 通过 `torch.utils.tensorboard` 完美支持。

### 核心能力一览

| 功能 | API | 解决的问题 |
|------|-----|-----------|
| **Scalars** | `add_scalar` / `add_scalars` | 损失、准确率趋势 |
| **Histograms** | `add_histogram` | 权重/梯度分布变化 |
| **Graph** | `add_graph` | 可视化模型结构 |
| **Images** | `add_image` / `add_images` | 查看输入/输出/特征图 |
| **HParams** | `add_hparams` | 超参数对比实验 |
| **Embeddings** | `add_embedding` | 高维特征降维可视化 |
| **Text** | `add_text` | 记录文本摘要 |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from tqdm import tqdm
from datetime import datetime

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

## 8.2 SummaryWriter 入门

`SummaryWriter` 是 TensorBoard 的日志记录器。它把数据写到 `log_dir` 目录，TensorBoard 从该目录读取并展示。

### 基础用法
```python
writer = SummaryWriter("runs/experiment_name")
writer.add_scalar("Loss/train", loss_value, global_step=epoch)
writer.close()  # 用完后关闭
```

### 目录命名规范
推荐使用时间戳或实验名区分不同 run：
```python
"runs/exp1_baseline"      # 手动命名
"runs/2024-01-01_14-30"   # 时间戳
"runs/{dataset}_{model}"  # 数据集+模型
```

In [ ]:
log_dir = Path("runs") / f"demo_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
writer = SummaryWriter(log_dir=str(log_dir))
print(f"Log directory: {log_dir}")

## 8.3 记录标量 (Scalars) — 损失和准确率

`add_scalar(tag, scalar_value, global_step)` 是最常用的 API，用于记录任何随时间变化的标量值。

### tag 命名技巧
使用 `/` 分隔可以创建层级分组：
- `"Loss/train"` — 训练损失，在 TensorBoard 中归入 "Loss" 组
- `"Loss/val"` — 验证损失
- `"Accuracy/train"` — 训练准确率
- `"LR"` — 学习率

### 同时记录多个标量
用 `add_scalars(main_tag, tag_scalar_dict, global_step)` 可以把多条曲线放到同一张图上对比。

In [ ]:
for step in range(100):
    train_loss = np.exp(-step / 30) + np.random.normal(0, 0.05)
    val_loss = np.exp(-step / 30) + 0.15 + np.random.normal(0, 0.03)
    train_acc = 0.1 + 0.85 * (1 - np.exp(-step / 30))
    val_acc = 0.1 + 0.75 * (1 - np.exp(-step / 30))

    writer.add_scalars("Loss", {"train": train_loss, "val": val_loss}, step)
    writer.add_scalars("Accuracy", {"train": train_acc, "val": val_acc}, step)

    lr = 0.001 * (1 - step / 100)
    writer.add_scalar("LR", lr, step)

print("Logged 100 steps of simulated scalars.  ✔")

## 8.4 记录直方图 (Histograms) — 权重与梯度

`add_histogram(tag, values, global_step)` 用于记录**张量的分布**。

### 关键用途
- **权重分布**：观察是否发生梯度消失/爆炸
- **梯度分布**：检查各层梯度流是否正常
- **激活值**：确认激活函数是否饱和（如 sigmoid 的梯度饱和）

### 小技巧
TensorBoard 的 Histogram 面板支持 **OFFSET** 模式，可以看到分布的**随时间变化**——这对于检测训练异常极其有用。

| 异常 | 权重分布特征 |
|------|-------------|
| **梯度消失** | 权重几乎不变，histogram 一致 |
| **梯度爆炸** | 权重值急剧增大，histogram 越来越宽 |
| **dead ReLU** | 大量权重集中在 0 附近 |

In [ ]:
class DemoNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(100, 50)
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model_demo = DemoNet()
opt_demo = optim.SGD(model_demo.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

for step in range(50):
    x = torch.randn(32, 100)
    y = torch.randint(0, 10, (32,))
    opt_demo.zero_grad()
    loss = criterion(model_demo(x), y)
    loss.backward()
    opt_demo.step()

    for name, param in model_demo.named_parameters():
        writer.add_histogram(f"Weights/{name}", param.data, step)
        if param.grad is not None:
            writer.add_histogram(f"Gradients/{name}", param.grad, step)

print("Logged 50 steps of weight/gradient histograms.  ✔")

## 8.5 记录模型计算图 (Graph)

`add_graph(model, input_to_model)` 可以自动追踪并可视化 PyTorch 模型的**计算图结构**。

### 注意
- `input_to_model` 必须是**一个 tensor 或 tuple of tensors**
- graph 只在调用时记录一次，通常放在训练开始前
- 支持双击节点展开查看内部细节

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(32 * 8 * 8, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

demo_model = SimpleCNN()
dummy_input = torch.randn(1, 3, 32, 32)
writer.add_graph(demo_model, dummy_input)
print("Model graph recorded to TensorBoard.  ✔")
print("Double-click nodes in the GRAPHS tab to expand.")

## 8.6 记录图像 (Images)

`add_image(tag, img_tensor, global_step)` 和 `add_images(tag, img_tensor, global_step)` 用于可视化图片。

### 典型使用场景
- 查看**原始输入** + **数据增强后**的样本
- 观察**特征图 (feature maps)**
- 查看**错误分类**的样本
- 显示**生成模型**的输出（如 GAN / VAE）

### 图像格式要求
TensorBoard 期望：
- `add_image`：`(C, H, W)` 或 `(batch, C, H, W)`
- `add_images`：`(N, C, H, W)`
- 像素值范围 [0, 1]（float）或 [0, 255]（uint8）

In [ ]:
cifar10_mean = (0.4914, 0.4822, 0.4465)
cifar10_std = (0.2470, 0.2435, 0.2616)

demo_ds = datasets.CIFAR10(root="data", train=True, download=True, transform=transforms.ToTensor())

images, labels = [], []
for i in range(8):
    img, lbl = demo_ds[i]
    images.append(img)
    labels.append(demo_ds.classes[lbl])

grid = torch.stack(images)
writer.add_images("CIFAR-10/Samples", grid, 0)
print("Logged 8 CIFAR-10 sample images.  ✔")

## 8.7 记录超参数 (HParams)

`add_hparams(hparam_dict, metric_dict)` 是 TensorBoard 的实验管理利器。

### 功能
- 对比**不同超参数组合**下的最终结果
- 在 HParams 仪表板中**并行视图**查看
- 支持**平行坐标图**（Parallel Coordinates Plot）发现参数间的关系
- 支持**散点图矩阵**（Scatter Plot Matrix）

In [ ]:
hparams = {
    "lr": 0.001,
    "batch_size": 128,
    "optimizer": "Adam",
    "use_dropout": True,
    "num_filters": 32,
}

metrics = {
    "hparam/accuracy": 0.854,
    "hparam/loss": 0.423,
    "hparam/epochs_to_best": 18,
}

writer.add_hparams(hparams, metrics)
print("HParams recorded. Open HParams tab to see comparison dashboard.  ✔")

## 8.8 在 Jupyter 中内嵌 TensorBoard

无需打开命令行，直接在 Notebook 里启动 TensorBoard！

### 方式 1：%tensorboard magic（推荐）
```python
%load_ext tensorboard
%tensorboard --logdir runs
```

### 方式 2：Python API 启动
```python
from tensorboard import notebook
notebook.start("--logdir runs")
```

In [ ]:
print("┌" + "─" * 58 + "┐")
print("│  [1;33m在 Jupyter 中启动 TensorBoard[0m                                     │")
print("│                                                              │")
print("│  %load_ext tensorboard                                      │")
print("│  %tensorboard --logdir runs --port 6006 --bind_all          │")
print("│                                                              │")
print("│  [90m或者访问 http://localhost:6006[0m                                │")
print("└" + "─" * 58 + "┘")

## 8.9 TensorBoard 命令行使用

如果不在 Jupyter 中，也可以在终端启动：

```bash
tensorboard --logdir runs --port 6006 --bind_all
```

常用参数：

| 参数 | 说明 |
|------|------|
| `--logdir` | 日志目录路径 |
| `--port` | 端口号 (默认 6006) |
| `--bind_all` | 允许外部访问（服务器场景） |
| `--reload_interval` | 刷新间隔，默认 30 秒 |
| `--samples_per_plugin` | 每个插件的采样数限制 |

### 如何选择 `--logdir`

```bash
# 查看单个实验
tensorboard --logdir runs/exp1

# 同时查看多个实验（通过父目录）
tensorboard --logdir runs        # 所有 experiment 一起显示

# 跨机器访问
tensorboard --logdir runs --host 0.0.0.0 --port 6006
```

## 8.10 实战：完整的 CIFAR-10 训练 + TensorBoard 全方位日志

我们将构建一个小型 CNN 并在训练过程中把**所有可观测信息**都记录到 TensorBoard。

### 记录清单
- **Scalars**: 每个 epoch 的 train/val loss 和 accuracy，学习率
- **Scalars (per batch)**: 每个 batch 的训练损失（细粒度曲线）
- **Histograms**: 每层的权重和梯度分布
- **Graph**: 模型计算图
- **Images**: 训练样本（含数据增强效果）、特征图、错误分类样本
- **Text**: 每个 epoch 的文本摘要
- **HParams**: 所有超参数 + 最终指标

In [ ]:
exp_name = f"cifar10_cnn_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
log_dir_exp = Path("runs") / exp_name
log_dir_exp.mkdir(parents=True, exist_ok=True)
print(f"Experiment log: {log_dir_exp}")

tb_writer = SummaryWriter(str(log_dir_exp))

### 8.10.1 数据准备

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

full_train = datasets.CIFAR10(root="data", train=True, download=True, transform=train_transform)
test_dataset = datasets.CIFAR10(root="data", train=False, download=True, transform=test_transform)

train_size = 45000
val_size = 5000
train_set, val_set = random_split(full_train, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=0, drop_last=True)
val_loader = DataLoader(val_set, batch_size=128, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0)

print(f"Train: {train_size:,} | Val: {val_size:,} | Test: {len(test_dataset):,}")

### 8.10.2 记录数据增强效果

In [ ]:
def denorm(img, mean=cifar10_mean, std=cifar10_std):
    img = img.clone()
    for t, m, s in zip(img, mean, std):
        t.mul_(s).add_(m)
    return img.clamp_(0, 1)

raw_train = datasets.CIFAR10(root="data", train=True, download=True, transform=train_transform)
aug_samples = torch.stack([denorm(raw_train[i][0]) for i in range(16)])
tb_writer.add_images("Data/Augmented_Samples", aug_samples, 0)
print("Augmented samples recorded to TensorBoard.")

### 8.10.3 模型定义 + 记录计算图

In [ ]:
class TB_CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
            nn.Dropout(0.3), nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = TB_CNN().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

dummy_graph = torch.randn(1, 3, 32, 32).to(device)
tb_writer.add_graph(model, dummy_graph)
print("Model graph recorded.  ✔")

### 8.10.4 带 TensorBoard 日志的训练循环

这是本单元的核心——展示如何在实际训练中全面使用 TensorBoard。

注意 `writer.add_scalar` 和 `writer.add_histogram` 的 `global_step` 参数：
- 它确定数据在时间轴上的位置
- 可以使用 epoch 数或全局 batch 计数

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25)

def train_one_epoch(model, loader, optimizer, criterion, epoch, writer, global_batch):
    model.train()
    total_loss, correct, total = 0, 0, 0
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}", leave=False)
    for data, target in pbar:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * data.size(0)
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += data.size(0)

        writer.add_scalar("Loss/train_batch", loss.item(), global_batch)
        writer.add_scalar("Accuracy/train_batch", pred.eq(target).float().mean().item(), global_batch)
        global_batch += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / total, correct / total, global_batch

@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    for data, target in loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        loss = criterion(output, target)
        total_loss += loss.item() * data.size(0)
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += data.size(0)
        all_preds.append(pred.cpu())
        all_labels.append(target.cpu())
    return total_loss / total, correct / total, torch.cat(all_preds), torch.cat(all_labels)

EPOCHS = 25
global_batch = 0
best_val_acc = 0.0

for epoch in range(EPOCHS):
    train_loss, train_acc, global_batch = train_one_epoch(
        model, train_loader, optimizer, criterion, epoch, tb_writer, global_batch,
    )
    val_loss, val_acc, val_preds, val_labels = validate(model, val_loader, criterion)
    scheduler.step()

    tb_writer.add_scalars("Loss", {"train": train_loss, "val": val_loss}, epoch)
    tb_writer.add_scalars("Accuracy", {"train": train_acc, "val": val_acc}, epoch)
    tb_writer.add_scalar("LR", optimizer.param_groups[0]["lr"], epoch)

    for name, param in model.named_parameters():
        tb_writer.add_histogram(f"Weights/{name}", param.data, epoch)
        if param.grad is not None:
            tb_writer.add_histogram(f"Gradients/{name}", param.grad, epoch)

    tb_writer.add_text(
        "Training_Summary",
        f"""| Epoch | Train Loss | Train Acc | Val Loss | Val Acc |
|:-----:|:----------:|:---------:|:--------:|:-------:|
| {epoch+1:5d} |   {train_loss:.4f}  |  {train_acc:.4f}  | {val_loss:.4f} | {val_acc:.4f} |""",
        epoch,
    )

    print(f"Epoch {epoch+1:3d}/{EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | LR: {optimizer.param_groups[0]['lr']:.2e}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), str(log_dir_exp / "best_model.pt"))
        print(f"  ➤ Best model saved! (Val Acc: {val_acc:.4f})")

print(f"\nTraining complete. Best Val Acc: {best_val_acc:.4f}")

### 8.10.5 记录错误分类样本

把验证集上预测错误的样本记录下来，方便分析模型弱点。

In [ ]:
model.load_state_dict(torch.load(log_dir_exp / "best_model.pt", map_location=device, weights_only=False))
model.eval()

raw_val = datasets.CIFAR10(root="data", train=True, download=True, transform=transforms.ToTensor())

test_transform_for_eval = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])
raw_val_trans = datasets.CIFAR10(root="data", train=True, download=True, transform=test_transform_for_eval)

val_indices = val_set.indices

mis_img = []
mis_true = []
mis_pred = []
for i, (data, target) in enumerate(val_loader):
    data, target = data.to(device), target.to(device)
    output = model(data)
    pred = output.argmax(dim=1)
    wrong_mask = pred != target
    if wrong_mask.any():
        for j in range(wrong_mask.sum().item()):
            if len(mis_img) >= 16:
                break
            idx = (i * 128 + wrong_mask.nonzero(as_tuple=True)[0][j].item()) % len(val_indices)
            orig_idx = val_indices[idx]
            orig_img, _ = raw_val[orig_idx]
            mis_img.append(orig_img)
            mis_true.append(f"True:{full_train.dataset.classes[target[wrong_mask][j].item()]}")
            mis_pred.append(f"Pred:{full_train.dataset.classes[pred[wrong_mask][j].item()]}")
        if len(mis_img) >= 16:
            break

if mis_img:
    grid = torch.stack(mis_img)
    tb_writer.add_images("Error_Analysis/Misclassified", grid, 0)
    for k, (t, p) in enumerate(zip(mis_true, mis_pred)):
        tb_writer.add_text("Error_Analysis/Labels", f"{k:2d}: {t} | {p}", k)
    print(f"Recorded {len(mis_img)} misclassified samples.")

### 8.10.6 记录特征图可视化

In [ ]:
@torch.no_grad()
def log_feature_maps(model, input_tensor, writer, tag_prefix, step):
    model.eval()
    features = {}
    def hook_fn(name):
        def hook(module, inp, out):
            features[name] = out.detach()
        return hook

    hooks = []
    for name, module in model.features.named_children():
        if isinstance(module, nn.ReLU) or isinstance(module, nn.MaxPool2d):
            continue
        hooks.append(module.register_forward_hook(hook_fn(name)))

    _ = model(input_tensor)
    for h in hooks:
        h.remove()

    for name, fmap in features.items():
        fmap = fmap[0].unsqueeze(1)
        n_show = min(16, fmap.size(0))
        fmap_show = fmap[:n_show].repeat(1, 3, 1, 1)
        fmap_show = (fmap_show - fmap_show.min()) / (fmap_show.max() - fmap_show.min() + 1e-8)
        writer.add_images(f"{tag_prefix}/{name}", fmap_show, step)

sample_input = torch.randn(1, 3, 32, 32).to(device)
log_feature_maps(model, sample_input, tb_writer, "Feature_Maps", 0)
print("Feature maps recorded to TensorBoard.")

### 8.10.7 记录 HParams + 最终测试结果

In [ ]:
test_loss, test_acc, _, _ = validate(model, test_loader, criterion)
print(f"Test Acc: {test_acc:.4f} ({test_acc*100:.2f}%)")

hparams_dict = {
    "lr": 0.05,
    "batch_size": 128,
    "optimizer": "SGD",
    "scheduler": "CosineAnnealingLR",
    "weight_decay": 5e-4,
    "epochs": EPOCHS,
    "model_type": "TB_CNN",
}

metrics_dict = {
    "hparam/best_val_acc": best_val_acc,
    "hparam/test_acc": test_acc,
    "hparam/test_loss": test_loss,
}

tb_writer.add_hparams(hparams_dict, metrics_dict)
print("HParams recorded.")

## 8.11 关闭 Writer

完成所有日志记录后，记得关闭 `SummaryWriter`。或者使用 `with` 语句自动管理。

In [ ]:
tb_writer.close()
print("TensorBoard writer closed.")

## 8.12 启动 TensorBoard 查看结果

运行下面 cell 在 Jupyter 中直接查看 TensorBoard，或运行上面的终端命令。

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs --port 6006

## 8.13 进阶技巧

### 技巧 1：为多个实验建立命名规范
```python
log_dir = f"runs/{model_name}_lr{lr}_bs{batch_size}_{timestamp}"
```
这样在 TensorBoard 的 `runs` 选择器中可以按中文/英文名快速区分。

### 技巧 2：用 `add_scalars` 做同图对比
- 训练损失 / 验证损失 放同一张图
- 不同层的梯度范数 放同一张图

### 技巧 3：定期记录而不是每 batch
每 batch 记录一次会让日志文件巨大。建议：
- **每 batch** 记录 `Loss/train_batch`（可选，需要细粒度调试时开）
- **每 epoch** 记录 histogram、images
- **每 N 个 epoch** 记录特征图和 embeddings

### 技巧 4：TensorBoard.dev 云端分享
```bash
tensorboard dev upload --logdir runs \
    --name "CIFAR-10 CNN Experiment" \
    --description "Comparing different architectures"
```
可以生成一个公开链接，方便与团队分享实验结果。

### 技巧 5：日志清理
TensorBoard 日志文件可能变得很大。定期清理：
```python
import shutil
shutil.rmtree("runs/old_experiment")  # 删除不需要的实验
```

## 8.14 单元小结

| 功能 | API | 最佳实践 |
|------|-----|---------|
| **Scalars** | `add_scalar` / `add_scalars` | 用 `/` 分层命名，train/val 同图 |
| **Histograms** | `add_histogram` | 每 epoch 记录，观察分布演变 |
| **Graph** | `add_graph` | 训练前记录一次 |
| **Images** | `add_images` | 记录样本、增强效果、错误分类 |
| **HParams** | `add_hparams` | 训练结束时记录，配合多实验对比 |
| **Text** | `add_text` | 记录 epoch 摘要、实验配置 |

### TensorBoard vs 手动绘图

| 方面 | `matplotlib` | TensorBoard |
|------|:-----------:|:-----------:|
| 设置成本 | 低 | 中 |
| 实时刷新 | ❌ 需要重新绘图 | ✅ 自动刷新 |
| 多实验对比 | ❌ 手动叠加 | ✅ 自动分组 |
| 权重/梯度分布 | ❌ 不支持 | ✅ Histogram |
| 计算图 | ❌ 不支持 | ✅ Graph 面板 |
| 超参数搜索 | ❌ 不支持 | ✅ HParams 面板 |
| 远端分享 | ❌ 需导出图片 | ✅ tensorboard.dev |
| 论文图 | ✅ 高质量原生 | ❌ 需截图处理 |

> **建议**：训练时用 TensorBoard 做实时监控，最终结果用 matplotlib 导出高质量论文图。两者互补，不是二选一。

### 思考题
1. 为什么建议只在每 epoch 记录 histogram 而非每 batch？
2. `add_scalar` 和 `add_scalars` 的区别是什么？什么时候用哪个？
3. 如果训练了 100 组超参数，如何在 TensorBoard 中找到最优的一组？
4. TensorBoard 的日志文件本质是什么格式？可以在 Notebook 里直接读取吗？